# Customer Churn Prediction EDA and Modeling

This notebook explores the telecom churn dataset, performs data preprocessing and feature engineering, trains multiple classifiers, and compares their performance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')


In [ ]:
df = pd.read_csv('../data/telecom_churn.csv')
print(df.head())
print('\nShape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())


In [ ]:
df.columns = [col.strip().lower() for col in df.columns]
print('Normalized columns:', list(df.columns))

# Clean TotalCharges
if 'totalcharges' in df.columns:
    df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')

# Create churn target
if 'churn' in df.columns:
    df['churn'] = df['churn'].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0, 'true': 1, 'false': 0})

# Feature engineering examples
if {'tenure', 'monthlycharges'}.issubset(df.columns):
    df['total_charge_to_tenure'] = df['monthlycharges'] * df['tenure']
    df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 60, 120], labels=['0-12', '13-24', '25-48', '49-60', '61+'])

print('\nAfter cleanup:\n', df.info())


In [ ]:
fig = px.pie(
    df['churn'].value_counts().reset_index().rename(columns={'index': 'Churn', 'churn': 'count'}),
    names='Churn',
    values='count',
    title='Churn Distribution'
)
fig.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='contract', hue='churn')
plt.title('Churn by Contract Type')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='churn', y='tenure')
plt.title('Tenure vs Churn')
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='monthlycharges', hue='churn', multiple='stack', kde=True)
plt.title('Monthly Charges Distribution by Churn')
plt.show()

fig = px.scatter(df, x='tenure', y='monthlycharges', color='churn', title='Tenure vs Monthly Charges by Churn')
fig.show()


In [ ]:
# Encode categorical variables for correlation matrix
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    df[col] = df[col].astype(str)

# Use only numeric columns for correlation
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

# Churn rate by contract customer segment
contract_summary = df.groupby('contract')['churn'].mean().reset_index()
fig = px.bar(contract_summary, x='contract', y='churn', title='Churn Rate by Contract')
fig.show()

# Payment method summary
payment_summary = df.groupby('paymentmethod')['churn'].mean().reset_index()
fig = px.bar(payment_summary, x='paymentmethod', y='churn', title='Churn Rate by Payment Method')
fig.show()


In [ ]:
target = 'churn'
X = df.drop(columns=[target], errors='ignore')
y = df[target].astype(int)

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Training shape:', X_train_processed.shape)
print('Test shape:', X_test_processed.shape)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values('ROC-AUC', ascending=False))


In [ ]:
importances = None
if 'Random Forest' in models:
    feature_names = preprocessor.get_feature_names_out()
    importances = models['Random Forest'].feature_importances_
    feat_importance = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)
    print(feat_importance)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_importance, x='importance', y='feature', palette='viridis')
    plt.title('Top 15 Feature Importances')
    plt.show()

# Save the best model
best_model = models['Random Forest']
joblib.dump(best_model, '../models/best_model.pkl')
print('Saved best model to ../models/best_model.pkl')


In [ ]:
new_customer = pd.DataFrame({
    'customerid': ['C999'],
    'gender': ['Female'],
    'seniorcitizen': [0],
    'partner': ['No'],
    'dependents': ['No'],
    'tenure': [12],
    'phoneservice': ['Yes'],
    'multiplelines': ['No'],
    'internetservice': ['Fiber optic'],
    'onlinesecurity': ['No'],
    'onlinebackup': ['Yes'],
    'deviceprotection': ['No'],
    'techsupport': ['No'],
    'streamingtv': ['Yes'],
    'streamingmovies': ['No'],
    'contract': ['Month-to-month'],
    'paperlessbilling': ['Yes'],
    'paymentmethod': ['Electronic check'],
    'monthlycharges': [75.0],
    'totalcharges': [900.0],
})

# Ensure consistent columns
new_customer.columns = [col.strip().lower() for col in new_customer.columns]
new_customer['totalcharges'] = pd.to_numeric(new_customer['totalcharges'], errors='coerce')

predicted = best_model.predict(preprocessor.transform(new_customer))
print('Prediction for new customer:', predicted[0])
